In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
DATASET_ROOT = r"E:\Dyploma\Carolingus\Carolingus\datasets\word_dataset"

In [4]:
def get_word_from_path(path: str) -> str:
    basename = os.path.basename(path).replace(".png", "")
    return basename.split("-")[1].lower()

get_word_from_path(r"E:\Dyploma\Carolingus\Carolingus\datasets\word_dataset\AUR_816_II_5-101 (text)\0-iussimus.png")


'iussimus'

In [9]:
word_to_images: dict[str, list] = {}

for subdir in os.listdir(DATASET_ROOT):
    subdir_path = os.path.join(DATASET_ROOT, subdir)
    if not os.path.isdir(subdir_path):
        continue
    
    for image in os.listdir(subdir_path):
        image_path = os.path.join(subdir_path, image)

        word = get_word_from_path(image_path)
        if word not in word_to_images:
            word_to_images[word] = [image_path]
        else:
            word_to_images[word].append(image_path)

len(word_to_images)

1767

In [10]:
RARITY_THRESHOLD = 2

rare_words = {}
for word, images in word_to_images.items():
    if len(images) <= RARITY_THRESHOLD:
        rare_words[word] = images

len(rare_words)

1382

In [11]:
def string_distance(str1: str, str2: str):
    len_str1 = len(str1) + 1
    len_str2 = len(str2) + 1

    distance_matrix = [[0] * len_str2 for _ in range(len_str1)]

    for i in range(len_str1):
        distance_matrix[i][0] = i
    for j in range(len_str2):
        distance_matrix[0][j] = j

    for i in range(1, len_str1):
        for j in range(1, len_str2):
            if str1[i - 1] == str2[j - 1]:
                cost = 0
            else:
                cost = 1

            distance_matrix[i][j] = min(
                distance_matrix[i - 1][j] + 1,
                distance_matrix[i][j - 1] + 1,
                distance_matrix[i - 1][j - 1] + cost,
            )

    return distance_matrix[-1][-1]

In [ ]:
STR_DIST_THRESH = 1

sim_words_2_image = word_to_images.copy()
words = list(word_to_images.keys())
deleted_words = set()

for i in range(len(words)):
    first_word = words[i]
    if first_word in deleted_words:
        continue

    for j in range(i + 1, len(words)):
        second_word = words[j]
        if second_word in deleted_words:
            continue
        dist = string_distance(first_word, second_word)

        if dist <= STR_DIST_THRESH:
            sim_words_2_image[first_word].extend(sim_words_2_image[second_word])
            del sim_words_2_image[second_word]
            deleted_words.add(second_word)

len(sim_words_2_image)

1405

In [15]:
RARITY_THRESHOLD = 2

rare_words = {}
n_rare_words_images = 0

for word, images in sim_words_2_image.items():
    if len(images) <= RARITY_THRESHOLD:
        rare_words[word] = images
        n_rare_words_images += len(images)

print("Number of rare words", len(rare_words))
print("Number of images of rare words", n_rare_words_images)

Number of rare words 1004
Number of images of rare words 1175


In [16]:
total_number_of_images = 0
for images in sim_words_2_image.values():
    total_number_of_images += len(images)

print(total_number_of_images)

5624
